# Explore the staged Green Taxi file (phase 2)
Exploring only. The pipeline never imports or runs this notebook.

In [3]:
import os
from pathlib import Path

import duckdb

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
staging = root / os.environ.get("STAGING_DIR", "data/raw")
file = staging / "green_tripdata_2025-01.parquet"
assert file.exists(), f"missing {file}: run make download PARTITION=2025-01"
src = f"read_parquet('{file}')"
con = duckdb.connect()

## Columns and types

In [4]:
con.sql(f'describe select * from {src}').df()[['column_name', 'column_type']]

,column_name,column_type
0,VendorID,INTEGER
1,lpep_pickup_datetime,TIMESTAMP
2,lpep_dropoff_datetime,TIMESTAMP
3,store_and_fwd_flag,VARCHAR
4,RatecodeID,BIGINT
5,PULocationID,INTEGER
6,DOLocationID,INTEGER
7,passenger_count,BIGINT
8,trip_distance,DOUBLE
9,fare_amount,DOUBLE


## Row count (expected value for A1)

In [5]:
con.sql(f'select count(*) as rows from {src}').df()

,rows
0,48326


## Missing values in the columns the output needs, and in ehail_fee

In [6]:
con.sql(f'''select count(*) filter (where lpep_pickup_datetime is null) as null_pickup,
  count(*) filter (where total_amount is null) as null_total_amount,
  count(*) filter (where ehail_fee is null) as null_ehail_fee
from {src}''').df()

,null_pickup,null_total_amount,null_ehail_fee
0,0,0,48326


## Value ranges

In [7]:
con.sql(f'''select min(lpep_pickup_datetime) as first_pickup, max(lpep_pickup_datetime) as last_pickup,
  min(total_amount) as min_total, max(total_amount) as max_total, round(sum(total_amount), 2) as sum_total,
  count(*) filter (where total_amount < 0) as negative, count(*) filter (where total_amount = 0) as zero
from {src}''').df()

,first_pickup,last_pickup,min_total,max_total,sum_total,negative,zero
0,2024-12-25 23:13:15,2025-02-05 18:46:24,-114.0,371.4,1093822.36,153,33


## Pickup dates outside the file's own month, and outside Jan to Mar 2025 (ADR 2)

In [8]:
con.sql(f'''select cast(lpep_pickup_datetime as date) as pickup_date, count(*) as rows
from {src} where lpep_pickup_datetime < '2025-01-01' or lpep_pickup_datetime >= '2025-02-01'
group by 1 order by 1''').df()

,pickup_date,rows
0,2024-12-25,1
1,2024-12-29,1
2,2024-12-31,4
3,2025-02-01,19
4,2025-02-03,6
5,2025-02-04,7
6,2025-02-05,5


## Natural key: is any row repeated on all columns?

In [9]:
con.sql(f'''select count(*) as rows, (select count(*) from (select distinct * from {src})) as distinct_rows from {src}''').df()

,rows,distinct_rows
0,48326,48326
